# 0. Imports

In [ ]:
import os
import re
import json
import time
from pathlib import Path
from tqdm import tqdm
from openai import OpenAI

# 1. Configuration & Paths

In [ ]:
BENCHMARK_PATH = Path("../data/processed/gold_benchmark.json")
RESULTS_DIR = Path("../data/results/multichoice")

MODELS_TO_EVALUATE = [
    "aminadaven/dictalm2.0-instruct:q2_k",
    "dicta-il/DictaLM-3.0-1.7B-Thinking:latest",
    "gemma3:4b",
    "hf.co/dicta-il/DictaLM-3.0-Nemotron-12B-Instruct-GGUF:Q4_K_M",
    "qwen3:8b"
]

local_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
    max_retries=0,
    timeout=90.0
)

# 2. Evaluation System Prompt

In [ ]:
MC_EVAL_SYSTEM_PROMPT = """אתה מומחה לניתוח טקסט, תרבות וסלנג ישראלי.
לפניך קטע משיר היפ הופ ישראלי, שורה ספציפית מתוכו, ו-4 אפשרויות הסבר למשמעותה (A, B, C, D).
עליך לבחור את התשובה הנכונה ביותר מתוך ההקשר התרבותי והלשוני.

השב אך ורק באובייקט JSON בפורמט הבא:
{
  "reasoning": "הסבר קצר ותמציתי (משפט אחד)",
  "selected_option": "A"
}"""

# 3. Helper Functions

In [ ]:
def format_eval_user_prompt(item: dict) -> str:
    options_text = "\n".join([f"{k}. {v}" for k, v in item["options"].items()])
    return f"""אמן: {item['artist']}
שיר: {item['song_title']}

הקשר מתוך השיר:
{item.get('context_stanza', '')}

השורה לבדיקה:
"{item['fragment']}"

אפשרויות:
{options_text}

איזו אפשרות היא ההסבר הנכון ביותר? ענה בפורמט JSON בלבד עם המפתח "selected_option" (A, B, C או D)."""

def extract_predicted_label(response_text: str) -> str:
    """Extracts A, B, C, or D cleanly from JSON or raw text."""
    clean_text = response_text.strip()
    if clean_text.startswith("```json"):
        clean_text = clean_text[7:-3].strip()
    elif clean_text.startswith("```"):
        clean_text = clean_text[3:-3].strip()
    
    try:
        data = json.loads(clean_text)
        pred = str(data.get("selected_option", "")).strip().upper()
        if pred in ["A", "B", "C", "D"]:
            return pred
    except Exception:
        pass
    
    # Fallback regex if the model outputted malformed JSON
    match = re.search(r'\b([A-D])\b', response_text.upper())
    return match.group(1) if match else "INVALID"

# 4. Evaluation Engine

In [ ]:
def evaluate_model_on_benchmark(
    model_name: str,
    benchmark_data: list[dict],
    client: OpenAI,
    results_dir: Path
):
    safe_model_name = model_name.replace(":", "_").replace("/", "_")
    output_file = results_dir / f"{safe_model_name}.json"
    
    evaluated_records = []
    completed_ids = set()
    
    # Resume checkpoint if partially run
    if output_file.exists():
        with output_file.open("r", encoding="utf-8") as f:
            evaluated_records = json.load(f)
            completed_ids = {r["id"] for r in evaluated_records}

    pending_items = [item for item in benchmark_data if item["id"] not in completed_ids]

    print(f"\n==================================================")
    print(f"Starting Evaluation: {model_name}")
    print(f"Total Items: {len(benchmark_data)} | Pending: {len(pending_items)} | Already Done: {len(completed_ids)}")
    print(f"Saving to: {output_file}")
    print(f"==================================================")

    correct_count = sum(1 for r in evaluated_records if r["is_correct"])

    for item in tqdm(pending_items, desc=f"Eval: {model_name}"):
        user_prompt = format_eval_user_prompt(item)
        
        raw_output = ""
        predicted_label = "ERROR"
        
        for attempt in range(3):
            try:
                response = client.chat.completions.create(
                    model=model_name,
                    messages=[
                        {"role": "system", "content": MC_EVAL_SYSTEM_PROMPT},
                        {"role": "user", "content": user_prompt}
                    ],
                    response_format={"type": "json_object"},
                    temperature=0.0  # Greedy decoding for strict reproducibility
                )
                raw_output = response.choices[0].message.content.strip()
                predicted_label = extract_predicted_label(raw_output)
                break
            except Exception as e:
                time.sleep(2)
                if attempt == 2:
                    raw_output = f"Inference Error: {str(e)}"
                    predicted_label = "ERROR"

        is_correct = (predicted_label == item["correct_label"])
        if is_correct:
            correct_count += 1

        record = {
            "id": item["id"],
            "fragment": item["fragment"],
            "correct_label": item["correct_label"],
            "predicted_label": predicted_label,
            "is_correct": is_correct,
            "raw_model_response": raw_output
        }
        
        evaluated_records.append(record)
        
        # Atomic save per question
        with output_file.open("w", encoding="utf-8") as f:
            json.dump(evaluated_records, f, ensure_ascii=False, indent=2)

    total_done = len(evaluated_records)
    acc = (correct_count / total_done * 100) if total_done > 0 else 0.0
    print(f"\nFinished {model_name}!")
    print(f"Accuracy: {acc:.2f}% ({correct_count}/{total_done})")
    
    return acc

# 5. Pipeline Execution Loop

In [ ]:
if not BENCHMARK_PATH.exists():
        raise FileNotFoundError(f"Cannot find benchmark file at: {BENCHMARK_PATH}")
        
    with BENCHMARK_PATH.open("r", encoding="utf-8") as f:
        benchmark_items = json.load(f)

    summary_results = {}
    
    for model in MODELS_TO_EVALUATE:
        try:
            acc = evaluate_model_on_benchmark(
                model_name=model,
                benchmark_data=benchmark_items,
                client=local_client,
                results_dir=RESULTS_DIR
            )
            summary_results[model] = f"{acc:.2f}%"
        except Exception as e:
            print(f"Skipping {model} due to error: {e}")
            summary_results[model] = "FAILED"

    print("\n" + "="*40)
    print("FINAL BENCHMARK LEADERBOARD (LOCAL MODELS)")
    print("="*40)
    for model, score in summary_results.items():
        print(f"{model:<25} | Accuracy: {score}")
    print("="*40)